In [1]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *

import matplotlib.pyplot as plt 

import pickle

from joblib import Parallel, delayed

import sys
from pathlib import Path

import subprocess

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

seed_num = 123


os.chdir(PROJECT_ROOT)

# Purpose

This notebook shows that the GREML implementation in our framework is close to that in the original GCTA implementation to a factor of $\sim 10^{-7}$. It takes around $10$ minutes to run with each simulation taking $2$ minutes. One needs to download the official GCTA software (https://yanglab.westlake.edu.cn/software/gcta/#Download) for these comparisons and to put the executable in the project root directory (one level above the notebook directory). The intermediate inputs for GCTA are not included in the repo because they can be quite large. We show that our implementation is similar with both AR1 and Realistic data.

In [2]:
def parse_gcta_output(filename):

    counter = 0
    start = 0
    end = 0
    lines = []
    with open(filename, 'r') as file:
        for line in file:
            # Process each line here
            # Lines include the newline character ('\n') at the end
            if 'Summary result of REML analysis:' in line:
                start = counter + 2
            if 'Sampling variance/covariance of the estimates of variance components:' in line:
                end = counter - 1
            counter += 1
            lines.append(line)
    
    
    cleaned_lines = [z.strip().replace('\t',' ') for z in lines[start:end]]
    h2_gcta = np.float64(cleaned_lines[-1].split(' ')[1]) # only need h2_gcta, SE estimation is out of scope.
    return h2_gcta

## AR1($\rho = 0.995$)

In [3]:
cleanup = True # Turns this off if you want to look at the intermediate files/logs. Otherwise, these are deleted.
np.random.seed(123)
num_sims = 5

h2_gctas = []
my_h2_gctas= []
for q in tqdm(range(num_sims)):
    i = np.random.choice(np.arange(0,1000,1),size = 1).item()
    
    rho = 0.995
    
    
    n,m = 5000,10000
    
    
    X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
    ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
    ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
    method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
    stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
    simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
    debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}
    
    list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
    params  = combine_all_dicts(list_of_dicts)
    
    to_run = dict()
    to_run['demonstration'] = params
    
    sim_key = 'demonstration'
    locals().update(to_run[sim_key])
    res_dict = dict()
    res_dict_raw = dict()
    
    #rhos = [0.995]
    counter = -1
    multithreading = True
    
    
    counter += 1
    key = str(rho)
    
    if make_ref_ldscores:
        ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = rho,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
        ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
        n_tilde = ref_data_gen.n
    else:
        ref_data_gen = None
        ref_ldscores = None
        n_tilde = None
    
        ref_mu2_hat = None
        ref_mu3_hat = None
        X_ref = None
    
    n_jobs = 1
    
    np.random.seed(seed_num + i)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
    
    X,b = my_data_gen.gen_X()
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = np.nan,regress_X_on_PC = False,regress_y_on_PC = False)
    
    X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    
    sim_id = str(i)
    GRM,GRM_id,pheno = my_data_gen.create_GCTA_inputs(X_tilde,y_tilde,sim_id = sim_id,output_files = True) # make GCTA files temporarily
    
    # Run GCTA
    path_to_gcta = 'bin/gcta64.exe'
    subprocess.run([path_to_gcta,'--reml',  '--grm-gz', sim_id+'_gcta',  '--pheno', sim_id+'_gcta.phen', '--out', sim_id + '_gcta'])
    filename = sim_id + '_gcta.log'
    h2_gcta = parse_gcta_output(filename)
    h2_gctas.append(h2_gcta)
    
    # Run our simulations from the beginning with the same seed
    res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)
    
    if cleanup:
        files_to_remove = [sim_id+'_gcta.phen',sim_id+'_gcta.grm.gz',sim_id+'_gcta.grm.id',sim_id+'_gcta.hsq',sim_id+'_gcta.log']
        for f in files_to_remove:
            os.remove(f)
    my_h2_gctas.append(res_dfs['h2_gcta'].item())
    



  0%|                                                                                            | 0/5 [00:00<?, ?it/s]INFO:root:Starting simulation 510
INFO:root:Finished simulation 510
 20%|████████████████▌                                                                  | 1/5 [02:17<09:09, 137.42s/it]INFO:root:Starting simulation 104
INFO:root:Finished simulation 104
 40%|█████████████████████████████████▏                                                 | 2/5 [04:33<06:50, 136.77s/it]INFO:root:Starting simulation 13
INFO:root:Finished simulation 13
 60%|█████████████████████████████████████████████████▊                                 | 3/5 [06:37<04:21, 130.81s/it]INFO:root:Starting simulation 62
INFO:root:Finished simulation 62
 80%|██████████████████████████████████████████████████████████████████▍                | 4/5 [08:49<02:11, 131.48s/it]INFO:root:Starting simulation 842
INFO:root:Finished simulation 842
100%|████████████████████████████████████████████████████████████████

In [4]:
out = pd.DataFrame([h2_gctas,my_h2_gctas]).T
out.columns = ['orig_h2_gcta','my_h2_gcta']
out['abs_diff'] = abs(out['orig_h2_gcta'] - out['my_h2_gcta'])

out

,orig_h2_gcta,my_h2_gcta,abs_diff
0,0.197167,0.197166,1.368487e-06
1,0.154575,0.154574,7.881940e-07
2,0.225286,0.225286,2.036259e-07
3,0.162357,0.162357,5.521997e-08
4,0.181031,0.181031,3.793392e-07


The absolute difference between implementations is around $\sim 10^{-7}$.

## Realistic

In [5]:
cleanup = True # Turns this off if you want to look at the intermediate files/logs. Otherwise, these are deleted.
np.random.seed(123)
num_sims = 5

h2_gctas = []
my_h2_gctas= []
for q in tqdm(range(num_sims)):
    i = np.random.choice(np.arange(0,1000,1),size = 1).item()
    
    rho = 0.995
    
    
    n,m = 5000,10000
    
    
    X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
    ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
    ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
    method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
    stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
    simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
    debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}
    
    list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
    params  = combine_all_dicts(list_of_dicts)
    
    to_run = dict()
    to_run['demonstration'] = params
    
    sim_key = 'demonstration'
    locals().update(to_run[sim_key])
    res_dict = dict()
    res_dict_raw = dict()
    
    #rhos = [0.995]
    counter = -1
    multithreading = True
    
    
    counter += 1
    key = str(rho)
    
    if make_ref_ldscores:
        ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = rho,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
        ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
        n_tilde = ref_data_gen.n
    else:
        ref_data_gen = None
        ref_ldscores = None
        n_tilde = None
    
        ref_mu2_hat = None
        ref_mu3_hat = None
        X_ref = None
    
    n_jobs = 1
    
    np.random.seed(seed_num + i)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)
    
    X,b = my_data_gen.gen_X()
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = np.nan,regress_X_on_PC = False,regress_y_on_PC = False)
    
    X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    
    sim_id = str(i)
    GRM,GRM_id,pheno = my_data_gen.create_GCTA_inputs(X_tilde,y_tilde,sim_id = sim_id,output_files = True) # make GCTA files temporarily
    
    # Run GCTA
    path_to_gcta = 'bin/gcta64.exe'
    subprocess.run([path_to_gcta,'--reml',  '--grm-gz', sim_id+'_gcta',  '--pheno', sim_id+'_gcta.phen', '--out', sim_id + '_gcta'])
    filename = sim_id + '_gcta.log'
    h2_gcta = parse_gcta_output(filename)
    h2_gctas.append(h2_gcta)
    
    # Run our simulations from the beginning with the same seed
    res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress).run_single_simulation_publication(i=i)
    
    if cleanup:
        files_to_remove = [sim_id+'_gcta.phen',sim_id+'_gcta.grm.gz',sim_id+'_gcta.grm.id',sim_id+'_gcta.hsq',sim_id+'_gcta.log']
        for f in files_to_remove:
            os.remove(f)
    my_h2_gctas.append(res_dfs['h2_gcta'].item())
    



  0%|                                                                                            | 0/5 [00:00<?, ?it/s]INFO:root:Starting simulation 510
INFO:root:Finished simulation 510
 20%|████████████████▌                                                                  | 1/5 [02:08<08:33, 128.40s/it]INFO:root:Starting simulation 985
INFO:root:Finished simulation 985
 40%|█████████████████████████████████▏                                                 | 2/5 [04:15<06:23, 127.67s/it]INFO:root:Starting simulation 239
INFO:root:Finished simulation 239
 60%|█████████████████████████████████████████████████▊                                 | 3/5 [06:20<04:13, 126.57s/it]INFO:root:Starting simulation 970
INFO:root:Finished simulation 970
 80%|██████████████████████████████████████████████████████████████████▍                | 4/5 [08:41<02:11, 132.00s/it]INFO:root:Starting simulation 953
INFO:root:Finished simulation 953
100%|████████████████████████████████████████████████████████████

In [6]:
out = pd.DataFrame([h2_gctas,my_h2_gctas]).T
out.columns = ['orig_h2_gcta','my_h2_gcta']
out['abs_diff'] = abs(out['orig_h2_gcta'] - out['my_h2_gcta'])

out

,orig_h2_gcta,my_h2_gcta,abs_diff
0,0.188891,0.188891,5.748374e-08
1,0.202973,0.202973,3.702928e-07
2,0.195414,0.195414,9.130903e-08
3,0.216857,0.216857,1.373779e-07
4,0.233735,0.233735,2.865611e-07


The absolute difference between the two implementations is also $\sim 10^{-7}$ like in the AR1 case.